# 01 · Explore the corpus

Survey the scan sources and the local cache. Everything reads the **single
cache / output root** resolved by `load_runtime()` — see `configs/pipeline.toml`
`[paths]`. Pull a book over WebDAV (dev) or off the mount (production) with the
configured source backend, then inspect a raw frame.

In [ ]:
import sys
from pathlib import Path

# project importable when the kernel isn't the project .venv
sys.path.insert(0, str(Path.cwd().parent))

from evilflowers_books_digitalizer.runtime import load_runtime
from evilflowers_books_digitalizer.cache import LocalCache

# ONE source of truth for cache_dir + output_dir (paths resolve to the project
# root regardless of the notebook's working directory) — this is the unified
# cache every notebook and the production runner share.
rt = load_runtime()
cache = LocalCache(rt.cache_dir)
print("cache :", rt.cache_dir)
print("output:", rt.output_dir)

In [ ]:
import pymupdf

def show_pdf_page(pdf_path, page=1, dpi=110):
    """Render one PDF page inline (for visual QA)."""
    from IPython.display import Image, display
    d = pymupdf.open(str(pdf_path))
    page = min(page, len(d) - 1)
    display(Image(data=d[page].get_pixmap(dpi=dpi).tobytes("png")))

def page_render_ms(pdf_path, dpi=150):
    """Mean per-page render time (ms) — the viewer-load-speed proxy."""
    import time
    d = pymupdf.open(str(pdf_path))
    ts = []
    for i in range(len(d)):
        t = time.time(); d[i].get_pixmap(dpi=dpi); ts.append((time.time() - t) * 1000)
    return sum(ts) / len(ts)

def cached_books():
    """(source, book_id) for every book staged in the local cache."""
    scans = rt.cache_dir / "scans"
    return sorted((p.parent.name, p.name) for p in scans.glob("*/*") if p.is_dir())

### Books already staged in the cache

In [ ]:
books = cached_books()
print(len(books), "book(s) in cache:")
for src, bid in books:
    n = len(cache.list_tiffs(src, bid))
    print(f"  {src:4} {bid}  ({n} frames)")

### Inspect a raw frame
Pick the first cached book and render a middle frame at screen resolution.

In [ ]:
import cv2, numpy as np
from IPython.display import Image, display

src, bid = books[0]
frames = cache.list_tiffs(src, bid)
mid = frames[len(frames) // 2]
img = cv2.imread(str(mid))
print(mid.name, "->", img.shape, "(h, w, c)")
small = cv2.resize(img, (img.shape[1] // 4, img.shape[0] // 4))
display(Image(data=cv2.imencode(".png", small)[1].tobytes()))

### Optional · pull a book from the configured source

Uncomment to stage a book from the WebDAV/filesystem backend named in
`[source]`. WebDAV needs `credentials.toml` + VPN; the filesystem backend reads
the mount. Both stage into the same cache directory used above.

In [ ]:
# from evilflowers_books_digitalizer import build_source
# source = build_source(rt.source, rt.source_keys[0])
# print("listing", rt.source_keys[0], "...")
# ids = source.list_books()[:5]
# print(ids)